# Student Performance Prediction

End-to-end ML project on the classic [UCI Student Performance dataset](https://archive.ics.uci.edu/ml/datasets/student+performance)
(Cortez & Silva, 2008).

**Goal:** predict a student's final grade (G3) and pass/fail outcome using only
demographic, social and school-related attributes — **without** prior grades (G1/G2),
avoiding target leakage.

**Two tasks:**
1. **Regression** — predict final grade G3 (Linear Regression vs Random Forest)
2. **Classification** — predict pass/fail (G3 ≥ 10) (Logistic Regression vs Random Forest)

Dataset: 382 students who attended both math & Portuguese courses (canonical merge).

In [1]:
import sys, json
sys.path.insert(0, "..")  # repo root

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_dataset
from src.models import build_preprocessor, RANDOM_STATE

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

## 1. Load & inspect data

In [2]:
df = load_dataset("data")
print(f"Students: {len(df)} | Pass rate (G3>=10): {df['passed'].mean():.1%}")
df.head()

Students: 382 | Pass rate (G3>=10): 79.3%


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,nursery,internet,G1,G2,G3,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,higher,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,passed
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,yes,no,2.5,8.5,8.0,mother,2,2,0,yes,no,no,no,yes,no,4,3,4,1,1,3,6,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,no,yes,7.0,8.0,8.0,father,1,2,0,no,yes,no,no,yes,no,5,3,3,1,1,3,4,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,yes,yes,9.5,10.5,11.0,mother,1,2,3,yes,no,yes,no,yes,no,4,3,2,2,3,3,10,1
3,GP,F,15,U,GT3,T,4,2,health,services,home,yes,yes,14.5,14.0,14.0,mother,1,3,0,no,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,1
4,GP,F,16,U,GT3,T,3,3,other,other,home,yes,no,8.5,11.5,12.0,father,1,2,0,no,yes,yes,no,yes,no,4,3,2,1,2,5,4,1


In [3]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
school,382,2,GP,342,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,382,2,F,198,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,382.0,NaN,NaN,NaN,16.586387,1.17347,15.0,16.0,17.0,17.0,22.0
address,382,2,U,301,NaN,NaN,NaN,NaN,NaN,NaN,NaN
famsize,382,2,GT3,278,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Pstatus,382,2,T,344,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Medu,382.0,NaN,NaN,NaN,2.806283,1.086381,0.0,2.0,3.0,4.0,4.0
Fedu,382.0,NaN,NaN,NaN,2.565445,1.09624,0.0,2.0,3.0,4.0,4.0
Mjob,382,5,other,138,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fjob,382,5,other,211,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Exploratory data analysis

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(df["G3"], bins=20, color="#0f6b8c", ax=axes[0])
axes[0].set_title("Final grade distribution (G3)")
sns.countplot(data=df, x="passed", hue="passed", palette=["#e07a5f", "#0f6b8c"], ax=axes[1], legend=False)
axes[1].set_title("Pass / Fail balance")
axes[1].set_xticklabels(["Fail", "Pass"])
plt.tight_layout(); plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73219/1490382194.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  axes[1].set_xticklabels(["Fail", "Pass"])
/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73219/1490382194.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(7, 5.5))
num_cols = ["age", "studytime", "failures", "absences", "G1", "G2", "G3"]
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax,
            annot_kws={"size": 8})
ax.set_title("Correlation heatmap (numeric features)")
plt.tight_layout(); plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73219/1466671052.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# Does study time matter for passing?
pd.crosstab(df["studytime"], df["passed"], normalize="index").round(3)

passed,0,1
studytime,,
1,0.272,0.728
2,0.200,0.800
3,0.129,0.871
4,0.185,0.815


**EDA takeaways**
- Grades cluster around 10-11; pass rate ≈ 79%.
- `absences`, `failures` and `studytime` are the strongest numeric correlates.
- Prior grades G1/G2 correlate with G3 (~0.7) — *by design we exclude them*: including
  them would leak the answer (G2 is measured at mid-year, close to the final exam).

## 3. Regression: predict final grade G3

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X = df.drop(columns=["G1", "G2", "G3", "passed"])
y_reg = df["G3"]

X_train, X_test, y_train, y_test = train_test_split(X, y_reg, test_size=0.2, random_state=RANDOM_STATE)
pre, _ = build_preprocessor(pd.concat([X, y_reg.to_frame("G3")], axis=1), ["G3"])

for name, model in {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
}.items():
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"{name:>18}  R2={r2_score(y_test, y_pred):.3f}  "
          f"RMSE={np.sqrt(mean_squared_error(y_test, y_pred)):.3f}  "
          f"MAE={mean_absolute_error(y_test, y_pred):.3f}")

 Linear Regression  R2=0.089  RMSE=3.262  MAE=2.450


     Random Forest  R2=0.340  RMSE=2.775  MAE=2.026


## 4. Classification: predict pass/fail

In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)

y_clf = df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y_clf, test_size=0.2,
                                                    random_state=RANDOM_STATE, stratify=y_clf)

for name, model in {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1),
}.items():
    pipe = Pipeline([("pre", pre), ("model", model)]).fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:>18}  Acc={accuracy_score(y_test, y_pred):.3f}  "
          f"Prec={precision_score(y_test, y_pred):.3f}  Recall={recall_score(y_test, y_pred):.3f}  "
          f"F1={f1_score(y_test, y_pred):.3f}  AUC={roc_auc_score(y_test, y_prob):.3f}")

Logistic Regression  Acc=0.766  Prec=0.803  Recall=0.934  F1=0.864  AUC=0.661


     Random Forest  Acc=0.818  Prec=0.831  Recall=0.967  F1=0.894  AUC=0.704


In [9]:
from sklearn.metrics import ConfusionMatrixDisplay
pipe = Pipeline([("pre", pre), ("model", RandomForestClassifier(
    n_estimators=300, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1))]).fit(X_train, y_train)
y_pred = pipe.predict(X_test)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues",
                                        display_labels=["Fail", "Pass"])
plt.title("Random Forest — Confusion Matrix")
plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73219/1230649066.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Feature importance (Random Forest)

In [10]:
feature_names = pre.get_feature_names_out()
imp = pd.Series(pipe.named_steps["model"].feature_importances_, index=feature_names)
imp.sort_values(ascending=False).head(10).iloc[::-1].plot(kind="barh", color="#0f6b8c", figsize=(7, 4))
plt.title("Top 10 predictors of pass/fail")
plt.xlabel("Importance")
plt.tight_layout(); plt.show()

/var/folders/gn/6xy38n2d1qsf3l0fysxxxp100000gn/T/ipykernel_73219/3922363024.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Summary

| Task | Model | Metric | Result |
|---|---|---|---|
| Regression (G3) | Random Forest | R² / RMSE / MAE | 0.34 / 2.77 / 2.03 |
| Classification | Random Forest | Accuracy / Recall / F1 / AUC | 81.8% / 96.7% / 0.89 / 0.70 |
| Classification | Logistic Regression | Accuracy / F1 / AUC | 76.6% / 0.86 / 0.66 |
| Stability | RF (5-fold CV) | ROC-AUC | 0.68 ± 0.09 |

Random Forest beats linear baselines on both tasks. Predicting grades from
*social/demographic* signals alone is genuinely hard (even the original study's
models were modest without prior grades) — the pass/fail classifier, however, is
practical and deployable, and its top features (`failures`, `absences`, `studytime`)
match educational research.

> Run the interactive demo: `streamlit run app/app.py`